# 第三階段：修準圓心，畫出結果

**輸入**：圖片路徑 + 第二階段找到的一批圓
**輸出**：修準後的一批圓（x, y, r，原圖尺度），並標記被邊界切到的圓；最後畫成一張圖存到你指定的路徑

## 怎麼用
1. 改「設定」那格的 `IMAGE_PATH` 和 `OUTPUT_PATH`
2. `CIRCLES` 留 `None`：自動讀第二階段存的結果（`.tmp/<圖片檔名>/detect/summary.json`）
3. 從上到下依序執行

## 為什麼要修？
第二階段是在**縮小的圖**上找圓，縮小圖上 1 個像素等於原圖好幾個像素，圓心會有誤差。
這一階段回到**原圖**，用原圖的清晰度再修一次。

## 怎麼修？（refine_center，四段）
1. **準備**：只在「大概的半徑」的 0.85～1.15 倍之間找外框（環帶，像甜甜圈）
2. **射線**：從圓心往 90 個方向各射一條線，每條線在環帶裡找邊緣最強的一點
3. **擬合**：丟掉最弱的 25% 點，用剩下的點算出最貼合的圓
4. **檢查**：修太多（超過半徑的 20%）就不相信，保留原本的

In [ ]:
import os, json, csv
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 如果上面出現 ModuleNotFoundError，先執行：
#   pip install opencv-python numpy matplotlib

## 設定（只要改這裡）

In [ ]:
IMAGE_PATH = "your_image.jpg"           # ← 改成你的圖片路徑
OUTPUT_PATH = "output/result.jpg"       # ← 畫好的圖要存在哪裡
CIRCLES = None                          # ← None = 自動讀第二階段的結果
TMP_DIR = ".tmp"

In [ ]:
name = os.path.splitext(os.path.basename(IMAGE_PATH))[0]
if CIRCLES is None:
    stage2 = os.path.join(TMP_DIR, name, "detect", "summary.json")
    CIRCLES = json.load(open(stage2))["circles"]
    print(f"從 {stage2} 讀到 {len(CIRCLES)} 個圓")
else:
    print(f"使用手動給的 {len(CIRCLES)} 個圓")

## 參數

In [ ]:
REFINE_PARAMS = {
    # 環帶：只在「大概的半徑」的幾倍之間找外框
    # 0.85～1.15 的意思是：半徑 100 的圓，只在離圓心 85～115 像素之間找
    # 圓內部的紋路（離圓心不到 85）完全不會被看到
    "RING": (0.85, 1.15),
    # 環帶裡每條射線取幾個檢查點
    "RING_STEPS": 31,
    # 從圓心往外射出幾條線（90 條 = 每 4 度一條）
    "N_RAYS": 90,
    # 至少要有幾成的射線可用，才進行修正（太多射線跑出圖外就不修）
    "MIN_RAYS_RATIO": 0.5,
    # 丟掉最弱的百分之幾的點（可能是外框的缺口或雜訊）
    "DROP_WEAKEST": 25,
    # 修完的圓心或半徑，如果變動超過原本半徑的幾倍，就不相信，保留原本的
    "MAX_CHANGE": 0.2,
    # 模糊 = 半徑 ÷ 這個數字。跟第二階段一樣是半徑的 1/20，換算回原圖大小
    "BLUR_DIVISOR": 20,
}

## 邊緣（跟前兩個階段一樣）

In [ ]:
def compute_edges(gray, blur):
    """邊緣強度 mag（跟前兩個階段一樣）"""
    b = cv2.GaussianBlur(gray, (0, 0), blur)
    gx = cv2.Sobel(b, cv2.CV_32F, 1, 0)
    gy = cv2.Sobel(b, cv2.CV_32F, 0, 1)
    return gx, gy, cv2.magnitude(gx, gy)

## 修準一個圓：refine_center
四段對應上面說明的「準備 → 射線 → 擬合 → 檢查」。

In [ ]:
def refine_center(mag, x, y, r, p=REFINE_PARAMS):
    """
    把一個「大概對」的圓修得更準。回傳 (新 x, 新 y, 新 r, 用了幾個點)。

    第 1 段：準備。在 0.85r～1.15r 之間平均取 31 個距離（環帶）
    第 2 段：射線。從圓心往 90 個方向各射一條線，
             每條線只在環帶裡找「邊緣最強」的那一點，當作外框上的點
    第 3 段：擬合。丟掉最弱的 25% 點，用剩下的點算出最貼合的圓（最小平方法）
    第 4 段：檢查。修太多就不相信，保留原本的
    """
    h, w = mag.shape

    # ---------- 第 1 段：準備 ----------
    lo, hi = p["RING"]
    rs = np.linspace(lo * r, hi * r, p["RING_STEPS"])

    # ---------- 第 2 段：射線 ----------
    pts, wts = [], []
    for a in np.linspace(0, 2 * np.pi, p["N_RAYS"], endpoint=False):
        X = np.round(x + rs * np.cos(a)).astype(int)
        Y = np.round(y + rs * np.sin(a)).astype(int)
        if not ((X >= 0) & (X < w) & (Y >= 0) & (Y < h)).all():
            continue                       # 這條線跑出圖外，跳過
        v = mag[Y, X]                      # 這條線上 31 個點的邊緣強度
        j = int(np.argmax(v))              # 最強的是第幾個
        pts.append((X[j], Y[j]))
        wts.append(v[j])

    if len(pts) < p["N_RAYS"] * p["MIN_RAYS_RATIO"]:
        return x, y, r, len(pts)           # 能用的點太少，不修

    # ---------- 第 3 段：擬合 ----------
    P = np.array(pts, float)
    W = np.array(wts)
    P = P[W > np.percentile(W, p["DROP_WEAKEST"])]
    # 圓的方程式 (x-a)² + (y-b)² = R² 可以改寫成 x² + y² = 2a·x + 2b·y + c
    # 每個點代入一次，得到很多條方程式，用最小平方法一次解出 a, b, c
    A = np.c_[2 * P[:, 0], 2 * P[:, 1], np.ones(len(P))]
    cx, cy, c = np.linalg.lstsq(A, (P ** 2).sum(axis=1), rcond=None)[0]
    cr = np.sqrt(c + cx ** 2 + cy ** 2)

    # ---------- 第 4 段：檢查 ----------
    if np.hypot(cx - x, cy - y) > p["MAX_CHANGE"] * r or abs(cr - r) > p["MAX_CHANGE"] * r:
        return x, y, r, len(P)
    return float(cx), float(cy), float(cr), len(P)

## 修準一批圓：refine_circles
對每個圓呼叫 refine_center，再檢查有沒有被邊界切到（圓心離邊界比半徑還近）。
中間資料存在 `.tmp/<圖片檔名>/refine/`：
- `before_red_after_green.jpg`：修正前（紅）與修正後（綠）
- `circles.csv`：每個圓修正前後的座標、移動了多少
- `summary.json`：參數和結果

In [ ]:
def refine_circles(image_path, circles, tmp_dir=".tmp", params=None):
    """
    輸入：
        image_path  圖片路徑
        circles     第二階段的結果：list of dict，每個至少有 x, y, r（原圖尺度）
    輸出：
        list of dict，每個圓：
            x, y, r    修準後的圓心和半徑（原圖尺度）
            score      第二階段的分數（原樣保留）
            shift      圓心移動了幾個像素
            partial    True = 圓有一部分在圖外，圓心可能不準
    """
    p = dict(REFINE_PARAMS, **(params or {}))
    name = os.path.splitext(os.path.basename(image_path))[0]
    out = os.path.join(tmp_dir, name, "refine")
    os.makedirs(out, exist_ok=True)

    gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise FileNotFoundError(image_path)
    H, W = gray.shape
    if not circles:
        return []

    # 在「原圖」上算邊緣。模糊跟第二階段一樣是半徑的 1/20
    r_typical = float(np.median([c["r"] for c in circles]))
    blur = max(1.0, r_typical / p["BLUR_DIVISOR"])
    mag = compute_edges(gray, blur)[2]

    results = []
    for c in circles:
        x, y, r, n_used = refine_center(mag, c["x"], c["y"], c["r"], p)
        # 圓心離邊界比半徑還近 → 圓有一部分在圖外
        partial = min(x, y, W - 1 - x, H - 1 - y) < r
        results.append({
            "x": round(x, 2), "y": round(y, 2), "r": round(r, 2),
            "score": c.get("score"),
            "shift": round(float(np.hypot(x - c["x"], y - c["y"])), 2),
            "points_used": int(n_used),
            "partial": bool(partial),
        })

    # ---------- 存中間資料 ----------
    # 修正前（紅）與修正後（綠）的對照圖
    img = cv2.imread(image_path)
    t = max(2, int(max(H, W) / 500))
    for before, after in zip(circles, results):
        cv2.circle(img, (int(round(before["x"])), int(round(before["y"]))),
                   int(round(before["r"])), (0, 0, 255), t)
        cv2.circle(img, (int(round(after["x"])), int(round(after["y"]))),
                   int(round(after["r"])), (0, 255, 0), t)
    cv2.imwrite(os.path.join(out, "before_red_after_green.jpg"), img)

    with open(os.path.join(out, "circles.csv"), "w", newline="") as f:
        wr = csv.writer(f)
        wr.writerow(["x_before", "y_before", "r_before", "x", "y", "r", "shift", "points_used", "partial"])
        for b, a in zip(circles, results):
            wr.writerow([b["x"], b["y"], b["r"], a["x"], a["y"], a["r"], a["shift"],
                         a["points_used"], a["partial"]])
    with open(os.path.join(out, "summary.json"), "w") as f:
        json.dump({"image": image_path, "params": p, "blur": blur, "circles": results}, f, indent=1)
    return results

## 執行修正

In [ ]:
refined = refine_circles(IMAGE_PATH, CIRCLES, tmp_dir=TMP_DIR)

shifts = [c_["shift"] for c_ in refined]
print(f"{len(refined)} 個圓，圓心平均移動 {np.mean(shifts):.1f} 像素（最多 {max(shifts):.1f}）")
print(f"完整的圓 {sum(not c_['partial'] for c_ in refined)} 個，"
      f"被邊界切到 {sum(c_['partial'] for c_ in refined)} 個")
print()
print(" 編號       x        y        r   partial")
for i, c_ in enumerate(refined):
    print(f"{i:4d} {c_['x']:8.1f} {c_['y']:8.1f} {c_['r']:8.1f}   {c_['partial']}")

## 畫出結果並存檔
- 紅色圓 = 完整的圓
- 橙色圓 = 被邊界切到的圓（圓心可能不準）
- 白色十字 = 圓心
- 數字 = 編號，對應上面表格

座標也會存成一個同名的 `.csv`，放在圖的旁邊。

In [ ]:
def draw_circles(image_path, circles, output_path, show_index=True):
    """
    把圓和圓心畫在原圖上，存到 output_path。
        紅色圓 = 完整的圓
        橙色圓 = 被邊界切到的圓（partial）
        白色十字 = 圓心
        數字 = 第幾個圓（對應 circles 的順序）
    """
    img = cv2.imread(image_path)
    H, W = img.shape[:2]
    t = max(2, int(max(H, W) / 400))                 # 線的粗細跟著圖的大小調整
    font = max(0.5, max(H, W) / 1500)
    for i, c in enumerate(circles):
        color = (0, 165, 255) if c.get("partial") else (0, 0, 255)
        center = (int(round(c["x"])), int(round(c["y"])))
        cv2.circle(img, center, int(round(c["r"])), color, t)
        cv2.drawMarker(img, center, (255, 255, 255), cv2.MARKER_CROSS, 6 * t, t)
        if show_index:
            cv2.putText(img, str(i), (center[0] + 3 * t, center[1] - 3 * t),
                        cv2.FONT_HERSHEY_SIMPLEX, font, (255, 255, 255), max(1, t // 2))
    folder = os.path.dirname(output_path)
    if folder:
        os.makedirs(folder, exist_ok=True)
    cv2.imwrite(output_path, img)
    return output_path


def show(path, title=None, width=10):
    """在 notebook 裡顯示一張存好的圖"""
    img = cv2.imread(path)
    if img is None:
        print("找不到圖：", path); return
    h, w = img.shape[:2]
    if title:
        print(title)   # 用 print 顯示標題（matplotlib 預設字型沒有中文）
    plt.figure(figsize=(width, width * h / w))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

In [ ]:
draw_circles(IMAGE_PATH, refined, OUTPUT_PATH)

csv_path = os.path.splitext(OUTPUT_PATH)[0] + ".csv"
with open(csv_path, "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(["index", "x", "y", "r", "partial"])
    for i, c_ in enumerate(refined):
        wr.writerow([i, c_["x"], c_["y"], c_["r"], c_["partial"]])

print("圖：", OUTPUT_PATH)
print("座標：", csv_path)
show(OUTPUT_PATH, "最後結果")